## Setup libraries and functions

In [ ]:
# Import key librarys
import pandas as pd
import numpy as np

In [ ]:
# Define tidy up initial df
def tidy(df):
    
    #### Fix usual issues with all strings
    
    # Capitalise columns
    df = df.map(lambda x: x.upper() if type(x) is str else x)

    # Strip whitespace
    df = df.map(lambda x: x.strip() if type(x) is str else x)

    # Remove parenthesis
    df = df.map(lambda x: x.replace('(', '') if type(x) is str else x)
    df = df.map(lambda x: x.replace(')', '') if type(x) is str else x)
    
    # Remove % signs
    df = df.map(lambda x: x.replace('%', '') if type(x) is str else x)
    
    # Remove linebreaks
    df = df.map(lambda x: x.replace('\n', '') if type(x) is str else x)

    # Replace annoying substrings
    df = df.map(lambda x: x.replace(' AND ', ' & ') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' – ', ' - ') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' / ', '/') if type(x) is str else x)
    df = df.map(lambda x: x.replace('/ ', '/') if type(x) is str else x)
    df = df.map(lambda x: x.replace(' /', '/') if type(x) is str else x)
        
    # Capitalise headers
    df.columns = df.columns.astype(str).str.upper()
    
    # Strip whitespace from headers
    df.columns = df.columns.str.strip()

    return df

# Further clean up of headers
def headertidy(df):
    
    # Remove parenthesis and % signs
    df.columns = df.columns.map(lambda x: x.replace('(', '') if type(x) is str else x)
    df.columns = df.columns.map(lambda x: x.replace(')', '') if type(x) is str else x)
    df.columns = df.columns.map(lambda x: x.replace('%', '') if type(x) is str else x)

    # Remove NOTE suffixes from column headers
    for i in range(10):

        df.columns = df.columns.map(lambda x: x.replace("[NOTE " + str(i) + "]", '') if type(x) is str else x)
    
    # Strip whitespace from headers
    df.columns = df.columns.map(lambda x: x.strip() if type(x) is str else x)

    return df

In [ ]:
# Define function to read in excel file
def readfile(year, sheet, rows):
    
        df = pd.read_excel('data/scotgov/' + year + '.xlsx', 
                                sheet_name=sheet,
                                skiprows=rows,
                                na_values = ['z', 'c','C','x', '#', '*', '.']
                                )
        return df

## Create list to store all data 

In [ ]:
frames = []

## Read in Scottish Government School Level data

In [ ]:
# Read in more recent data

# Create list of years
years = ['2021', '2022','2023', '2024']

sheets = {'2024': ['2024 School Level Statistics', 1], 
          '2023': ['2023 School Level Statistics', 1],
          '2022': ['2022 School Level Statistics', 1],
          '2021': ['2021 School Level Statistics', 1],
          '2020': ['2020', 1],
          '2019': ['2019', 1]}

# Loop through sheets dictionary
for y, s in sheets.items():
    
    # Read in df
    wdf = readfile(y, s[0], s[1])

    # Tidy df
    wdf = tidy(wdf)
    
    # Further tidy df headers
    wdf = headertidy(wdf)
    
    # Rename columns
    wdf = wdf.rename(columns={'SCHOOL NAME': 'SCHOOL',
                                'LOCAL AUTHORITY': 'LA', 
                                'SCHOOL TYPE': 'TYPE',
                                'TEACHERS FULL TIME EQUIVALENT': 'TEACHERS',
                                'FTE TEACHERS': 'TEACHERS',
                                'PUPIL ROLL': 'PUPILS',
                                'PUPIL ROLL1': 'PUPILS',
                                'FTE TEACHERS': 'TEACHERS',
                                'PUPILS WITH AN ADDITIONAL SUPPORT NEED RECORDED': 'ASN PUPILS'})
    
    # Select Secondary Schools
    wdf = wdf.loc[wdf['TYPE'] == 'SECONDARY']
    
    # Select key columns
    wdf = wdf[['LA', 'SCHOOL', 'PUPILS', 'ASN PUPILS']]
    
#     # Add column for year
#     wdf['YEAR'] = y
    
    # Melt data into long format
    ldf = pd.melt(wdf, id_vars=['LA', 'SCHOOL'],
                    value_vars=wdf.columns, var_name='VARIABLE', value_name='COUNT')
    
    # Add column for year
    ldf['YEAR'] = y
        
    # Append to list of df (with year)
    frames.append(ldf)

In [ ]:
ldf.to_csv('test.csv')

## Read in SQA Additional Arrangements data

In [ ]:
# Create dictionary of years, with sheet names and rows to skip
sheets = {2025: ['25428 2025', 1], 
          2024: ['25428 2024', 1],
          2023: ['25428 2023', 1],
          2022: ['25428 2022', 2],
          2019: ['25428 2019', 1]}

# Loop through sheets dictionary
for y, s in sheets.items():
    
    # Read in excel file
    wdf = pd.read_excel('data/sqa/FOI25 26 105 Information.xlsx',
                                    sheet_name=s[0],
                                    skiprows=s[1],
                                    na_values = ['[c]', 'Not Applicable'],
                                    skipfooter = 1
                                    )
    
    # Tidy df
    wdf = tidy(wdf)

    # Dictionary to map LA replacements
    headers = {'CENTRE TYPE': 'TYPE',
                'AUTHORITY': 'LA',
                'CENTRE': 'SCHOOL',
                'HIGHER': 'H',
                'NAT 5': 'N5'}

    # Replace column headers using dictionary
    wdf.rename(columns=headers, inplace=True)
    
    # Replace value in null LA cells with value from TYPE column
    wdf['LA'] = wdf.apply(lambda x: x['TYPE'] if pd.isnull(x['LA']) else x['LA'], axis=1)
    
    # Remove several prefixes / suffixes to tidy up
    wdf['TYPE'] = wdf['TYPE'].str.removeprefix('EDUCATION AUTHORITY - ')
    wdf['TYPE'] = wdf['TYPE'].str.removeprefix('INDEPENDENT - ')
    wdf['LA'] = wdf['LA'].str.removesuffix(' COUNCIL EDUCATION DEPARTMENT')
    # Remove two suffixes to tidy up from copying over TYPE values    
    wdf['LA'] = wdf['LA'].str.removesuffix(' - SECONDARY SCHOOL')
    wdf['LA'] = wdf['LA'].str.removesuffix(' - SPECIAL SCHOOL')
    
    # Select Secondary Schools
    wdf = wdf.loc[wdf['TYPE'] == 'SECONDARY SCHOOL']
    
    # Change into long format
    ldf = pd.melt(wdf, id_vars =['LA', 'SCHOOL'], 
        value_vars = ['AH', 'H', 'N5'],
                var_name ='VARIABLE', value_name ='COUNT')
    
    # Add AA (additional arrangements) identifier into VARIABLE column cells
    ldf['VARIABLE'] = ldf['VARIABLE'] + ' AA'
    
    # Add in column with year
    ldf['YEAR'] = y
    
    # Append to df list
    frames.append(ldf)

In [ ]:
ldf.head()

## Read in SQA Exam data

In [ ]:
# Create dictionary of years, with sheet names and rows to skip
sheets = {2019: 'Attainment by Centre 2019.csv',
            2022: 'Attainment by Centre 2022.csv',
            2023: 'Attainment by Centre 2023.csv',
            2024: 'Attainment by Centre 2024.csv',
            2025: 'Attainment by Centre 2025.csv'}

# Loop through sheets dictionary
for y, f in sheets.items():
    
    # Read in excel file
    wdf = pd.read_csv('data/sqa/' + f,
                                    skiprows=1,
                                    na_values = '[c]'
                                    )

    # Tidy df
    wdf = tidy(wdf)
    
    # Dictionary to map LA replacements
    headers = {'CENTRE': 'CODE',
                'CENTRE TYPE': 'TYPE',
                'EDUCATION_AUTHORITY': 'LA',
                'CENTRE NAME': 'SCHOOL'}

    # Replace column headers using dictionary
    wdf.rename(columns=headers, inplace=True)

    # Replace value in null LA cells with value from TYPE column
    wdf['LA'] = wdf.apply(lambda x: x['TYPE'] if pd.isnull(x['LA']) else x['LA'], axis=1)
    
    # Remove several prefixes / suffixes to tidy up
    wdf['TYPE'] = wdf['TYPE'].str.removeprefix('EDUCATION AUTHORITY - ')
    wdf['TYPE'] = wdf['TYPE'].str.removeprefix('INDEPENDENT - ')
    wdf['LA'] = wdf['LA'].str.removesuffix(' COUNCIL EDUCATION DEPARTMENT')
    # Remove two suffixes to tidy up from copying over TYPE values    
    wdf['LA'] = wdf['LA'].str.removesuffix(' - SECONDARY SCHOOL')
    wdf['LA'] = wdf['LA'].str.removesuffix(' - SPECIAL SCHOOL')
    
    # Select Secondary Schools
    wdf = wdf.loc[wdf['TYPE'] == 'SECONDARY SCHOOL']
    
    # Select key columns and groupby sum of entries per level
    gdf = wdf.groupby(['LA', 'SCHOOL', 'LEVEL'], as_index = False)['ENTRIES'].sum()
    
    # Dictionary to map LA replacements
    levels = {'ADVANCED': 'AH',
                'HIGHER': 'H',
                'NATIONAL 5': 'N5'}

    # Replace level values using dictionary
    gdf = gdf.replace({'LEVEL': levels})
    
        
    # Dictionary to map LA replacements
    headers = {'LEVEL':'VARIABLE',
                'ENTRIES': 'COUNT'}
    
    # Rename LEVEL column to VARIABLE
    gdf.rename(columns=headers, inplace=True)
    
    # Add E (Entries) identifier into VARIABLE column cells
    gdf['VARIABLE'] = gdf['VARIABLE'] + ' E'
    
    # Add in column with year
    gdf['YEAR'] = y
    
    # Append to df list
    frames.append(gdf)

In [ ]:
gdf.head()

## Test output

In [ ]:
# Concat list of dfs together
tdf = pd.concat(frames)

# Drop any rows with NaNs
tdf = tdf.dropna()

In [ ]:
# Remove several prefixes to tidy up
tdf['LA'] = tdf['LA'].str.removeprefix('THE CITY OF ')
tdf['LA'] = tdf['LA'].str.removeprefix('CITY OF ')
# Remove several suffixes to tidy up  
tdf['LA'] = tdf['LA'].str.removesuffix(' CITY')

# Dictionary to map LAreplacements
las = {'WESTERN ISLES': 'COMHAIRLE NAN EILEAN SIAR'}
las = {'NA H-EILEANAN SIAR': 'COMHAIRLE NAN EILEAN SIAR'}

# Replace level values using dictionary
tdf = tdf.replace({'LA': las})

In [ ]:
tdf.head()

In [ ]:
# Export to csv
tdf.to_csv('./csvs/aa.csv', index=False)

## Pivot data

In [ ]:
pdf = tdf.pivot(index=['LA', 'SCHOOL'], columns=['YEAR', 'VARIABLE'], values='COUNT')

In [ ]:
pdf.head()